# 分词器 (Tokenizer) 总览与实战

> 把任意文本切成离散单元（Token）的算法。LLM 的性能、上下文长度视角以及跨语言能力都由它决定。

本笔记本是分词器系列的**导引地图**。我们将对比常用的四种算法，并提供一个统一的接口示例。

## 1. 算法全景地图

| 算法 | 代表模型 | 核心逻辑 | 特点 | 对应子笔记本 |
| --- | --- | --- | --- | --- |
| **BPE (教学版)** | - | 字符合并 | 简单易懂，合并频次最高项 | [01 · BPE](01_bpe.ipynb) |
| **Byte-level BPE** | GPT-2/3, RoBERTa | 字节合并 | **0 OOV**，主流 GPT 选型 | [02 · BBPE](02_byte_level_bpe.ipynb) |
| **WordPiece** | BERT, DistilBERT | 似然最大化 | 续接前缀 `##`，互信息驱动 | [03 · WordPiece](03_wordpiece.ipynb) |
| **Unigram LM** | LLaMA, Qwen, T5 | 自顶向下裁剪 | Viterbi 解码，当前 SOTA 选型 | [04 · Unigram](04_unigram.ipynb) |

---

## 2. 核心对比

### 2.1 训练目标对比
- **BPE/BBPE**：寻找“出现次数最多”的组合。
- **WordPiece**：寻找“最显著相关”（即合并后能让语言模型似然率提升最大）的组合。
- **Unigram**：从一个全集中，逐步删除“最不重要”的词，直到剩下目标数量。

### 2.2 编码策略对比
- **BPE/WordPiece**：贪心（Greedy）。BPE 按合并规则顺序，WordPiece 按最长匹配在前。
- **Unigram**：全局最优。使用 Viterbi 算法寻找概率最大的分段方案。

---

## 3. 在本项目中如何集成？

本项目提供了统一的抽象基类 `BaseTokenizer`，并在 [`core/tokenizer/__init__.py`](../../core/tokenizer/__init__.py) 中通过工厂函数进行管理。

### 实战演示：训练一个 BBPE Tokenizer

In [ ]:
import sys
import os
# 将项目根目录加入 path (假设在 docs/01_tokenizer 下运行)
sys.path.append(os.path.abspath("../../"))

from core.tokenizer import build_tokenizer

# 1. 创建工厂实例
tokenizer = build_tokenizer("byte_bpe", vocab_size=256 + 100)

# 2. 准备语料
corpus = [
    "I love deep learning.",
    "BPE is a simple but powerful algorithm.",
    "Tokenizer is the first step of LLM."
]

# 3. 训练
tokenizer.train(corpus, verbose=True)

# 4. 编码与解码
text = "I love BPE Tokenizer!"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(f"\n原文: {text}")
print(f"IDs: {ids}")
print(f"解码结果: {decoded}")

## 4. 后续步骤建议

1. **零基础入门**：先读 [01 · BPE（教学版）](01_bpe.ipynb) 弄懂核心循环。
2. **理解主流 GPT**：阅读 [02 · Byte-level BPE](02_byte_level_bpe.ipynb) 理解为何没有 `<UNK>`。
3. **理解 LLaMA/Qwen**：阅读 [04 · Unigram LM](04_unigram.ipynb) 学习目前大模型最主流的实现方式。

---
> [README.md](README.md) 是分词器模块的文档入口。